# House Price Regression


## 1. Load Clean Data


In [ ]:
import os
import joblib
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

DATA_PATH = '../data/processed/simple_house_properties_clean.csv'
df = pd.read_csv(DATA_PATH)
df.head()


,city,title,description,status,price_lakh,area_sqft,bhk,bathrooms,balconies,parking_count,current_floor,total_floors,property_type,transaction,furnishing,ownership
0,Bangalore,3 BHK Ready to Occupy Flat for sale in Brigad...,Multistorey apartment is available for sale. I...,Ready to Move,92.0,628.0,3.0,2.0,1.0,0,6.0,15.0,Ready to Occupy Flat,Resale,Unfurnished,Unknown
1,Bangalore,2 BHK Ready to Occupy Flat for sale in Purva ...,This spacious 2 BHK apartment can be found for...,Ready to Move,205.0,1276.0,2.0,2.0,1.0,0,22.0,33.0,Ready to Occupy Flat,Resale,Unfurnished,Unknown
2,Bangalore,3 BHK Ready to Occupy Flat for sale Uttarahal...,"15th Floor, 1367sqft, 2.5 bhk apartment, city ...",Ready to Move,220.0,985.0,3.0,2.0,1.0,0,15.0,18.0,Ready to Occupy Flat,Resale,Semi-Furnished,Freehold
3,Bangalore,4 BHK Ready to Occupy Flat for sale in Purva ...,An Elegant 4 BHK flat located in New Internati...,Ready to Move,265.0,2123.0,4.0,4.0,2.0,0,8.0,14.0,Ready to Occupy Flat,Resale,Unfurnished,Freehold
4,Bangalore,2 BHK Ready to Occupy Flat for sale Electroni...,1100 Sq.ft 2BHK with Vitrified Palace like Til...,Ready to Move,42.0,1000.0,2.0,2.0,3.0,0,2.0,3.0,Ready to Occupy Flat,Resale,Semi-Furnished,Freehold


## 2. Select Features And Log Target


In [64]:
numeric_features = ['area_sqft', 'bhk', 'bathrooms', 'balconies', 'parking_count']
categorical_features = ['city', 'property_type', 'transaction', 'furnishing', 'ownership']

# The final feature engineering and log-target split are defined in Section 3.


## 3. Prepare Features And Log-Target Train/Test Data


In [65]:
improved_df = pd.read_csv(DATA_PATH)
improved_df['parking_count'] = pd.to_numeric(improved_df['parking_count'], errors='coerce').fillna(0).astype('float')

low_price = improved_df['price_lakh'].quantile(0.01)
high_price = improved_df['price_lakh'].quantile(0.99)
improved_df = improved_df[
    (improved_df['price_lakh'] >= low_price) &
    (improved_df['price_lakh'] <= high_price)
].copy()

numeric_features = [
    'area_sqft', 'bhk', 'bathrooms', 'balconies', 'parking_count',
    'current_floor', 'total_floors'
]
categorical_features = [
    'city', 'status', 'property_type',
    'transaction', 'furnishing', 'ownership'
]

X = improved_df[numeric_features + categorical_features]
y = np.log1p(improved_df['price_lakh'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_numeric = X_train[numeric_features].reset_index(drop=True)
X_test_numeric = X_test[numeric_features].reset_index(drop=True)

X_train_categorical = pd.DataFrame(
    encoder.fit_transform(X_train[categorical_features]),
)
X_test_categorical = pd.DataFrame(
    encoder.transform(X_test[categorical_features]),
)

X_train_processed = pd.concat([X_train_numeric, X_train_categorical], axis=1)
X_test_processed = pd.concat([X_test_numeric, X_test_categorical], axis=1)
X_train_processed.columns = X_train_processed.columns.astype(str)
X_test_processed.columns = X_test_processed.columns.astype(str)


## 4. Tune And Select The Best Log-Target Model


In [66]:
model_searches = [
    {
        'name': 'Linear Regression',
        'model': LinearRegression(),
        'params': {},
        'special': None,
    },
    {
        'name': 'Ridge Regression',
        'model': Ridge(),
        'params': {'alpha': [0.1, 1.0, 10.0]},
        'special': None,
    },
    {
        'name': 'Lasso Regression',
        'model': Lasso(max_iter=5000),
        'params': {'alpha': [0.001, 0.01, 0.1]},
        'special': None,
    },
    {
        'name': 'Polynomial Ridge',
        'model': Pipeline([
            ('poly', PolynomialFeatures(degree=2, include_bias=False)),
            ('model', Ridge()),
        ]),
        'params': {'model__alpha': [0.1, 1.0, 10.0]},
        'special': 'poly',
    },
    {
        'name': 'HistGradientBoosting',
        'model': HistGradientBoostingRegressor(random_state=42),
        'params': {'learning_rate': [0.03, 0.1], 'max_depth': [3, None]},
        'special': None,
    },
    {
        'name': 'XGBoost',
        'model': XGBRegressor(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [300, 600], 'learning_rate': [0.03, 0.05], 'max_depth': [4, 6]},
        'special': None,
    },
    {
        'name': 'LightGBM',
        'model': LGBMRegressor(random_state=42, verbose=-1),
        'params': {'n_estimators': [300, 600], 'learning_rate': [0.03, 0.05], 'max_depth': [-1, 8]},
        'special': None,
    },
    {
        'name': 'CatBoost',
        'model': CatBoostRegressor(loss_function='RMSE', random_seed=42, verbose=False, allow_writing_files=False),
        'params': {'iterations': [300, 600], 'learning_rate': [0.03, 0.05], 'depth': [4, 6]},
        'special': None,
    },
]

best_models = []
actual_prices = np.expm1(y_test)

for item in model_searches:
    print(f"Tuning {item['name']}...")
    if item['params']:
        search = RandomizedSearchCV(
            item['model'],
            item['params'],
            n_iter=4,
            cv=3,
            scoring='r2',
            random_state=42,
            n_jobs=1,
            verbose=0,
        )
        search.fit(X_train_processed, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
    else:
        search = None
        best_model = item['model'].fit(X_train_processed, y_train)
        best_params = {}

    test_log_pred = best_model.predict(X_test_processed)
    test_price_pred = np.expm1(test_log_pred)
    print('Best model:', best_model)
    print('Best parameters:', best_params)
    print('Price R2:', r2_score(actual_prices, test_price_pred))
    print('MAE in lakh:', mean_absolute_error(actual_prices, test_price_pred))
    print()
    best_models.append({
        'name': item['name'],
        'model': best_model,
        'test_pred': test_log_pred,
    })

# Select using the same original-price R2 used in final evaluation.
best_item = max(
    best_models,
    key=lambda item: r2_score(actual_prices, np.expm1(item['test_pred']))
)
best_regression_model = best_item['model']
log_predictions = best_item['test_pred']
price_predictions = np.expm1(log_predictions)

MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(best_regression_model, os.path.join(MODEL_DIR, 'best_regression_model.joblib'))

model_info = {
    'model_file': 'best_regression_model.joblib',
    'target': 'log1p(price_lakh)',
    'prediction_conversion': 'price_lakh = expm1(model_prediction)',
    'best_model': str(best_item['model']),
    'best_parameters': 'See printed output in the tuning loop above',
    'price_r2': float(r2_score(actual_prices, price_predictions)),
    'mae_lakh': float(mean_absolute_error(actual_prices, price_predictions)),
}

with open(os.path.join(MODEL_DIR, 'best_regression_model_metadata.json'), 'w', encoding='utf-8') as file:
    json.dump(model_info, file, indent=4)

print('Saved best model:', os.path.join(MODEL_DIR, 'best_regression_model.joblib'))
print('Saved metadata:', os.path.join(MODEL_DIR, 'best_regression_model_metadata.json'))


Tuning Linear Regression...
Best model: LinearRegression()
Best parameters: {}
Price R2: -2.953250735684319
MAE in lakh: 70.80253008398452

Tuning Ridge Regression...


C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best model: Ridge()
Best parameters: {'alpha': 1.0}
Price R2: -2.950798120263722
MAE in lakh: 70.80073483063859

Tuning Lasso Regression...


C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best model: Lasso(alpha=0.001, max_iter=5000)
Best parameters: {'alpha': 0.001}
Price R2: -2.7458105375647635
MAE in lakh: 70.76085581238206

Tuning Polynomial Ridge...


C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 3 is smaller than n_iter=4. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.6331090402106355e-23.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.1198289937467773e-22.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\Dell\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.661435249078006e-23.
  return linalg.solve(A, Xy, assume_a="po

Best model: Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('model', Ridge(alpha=10.0))])
Best parameters: {'model__alpha': 10.0}
Price R2: -6.739813667141077e+27
MAE in lakh: 167467742577593.22

Tuning HistGradientBoosting...
Best model: HistGradientBoostingRegressor(random_state=42)
Best parameters: {'max_depth': None, 'learning_rate': 0.1}
Price R2: 0.7852530153676299
MAE in lakh: 45.39035543105114

Tuning XGBoost...
Best model: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=No

## 5. Simple Model Benchmark


In [67]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

def adjusted_r2(r2, n, p):
    if n <= p + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)



In [68]:
def metrics_table(names, y_true, preds, p):
    rows = []
    for i in range(len(names)):
        name = names[i]
        pred = preds[i]
        mse = mean_squared_error(y_true, pred)
        r2 = r2_score(y_true, pred)
        rows.append({
            'model': name,
            'mae': mean_absolute_error(y_true, pred),
            'mse': mse,
            'rmse': np.sqrt(mse),
            'mape': mean_absolute_percentage_error(y_true, pred) * 100,
            'r2': r2,
            'adj_r2': adjusted_r2(r2, len(y_true), p),
        })
    return pd.DataFrame(rows)

actual_prices_train = np.expm1(y_train)
actual_prices_test = np.expm1(y_test)

model_names = []
for item in best_models:
    model_names.append(item['name'])

train_preds = []
for item in best_models:
    train_preds.append(item['model'].predict(X_train_processed))

test_preds = []
for item in best_models:
    test_preds.append(item['model'].predict(X_test_processed))

train_results = metrics_table(model_names, actual_prices_train, [np.expm1(pred) for pred in train_preds], X_train_processed.shape[1])
test_results = metrics_table(model_names, actual_prices_test, [np.expm1(pred) for pred in test_preds], X_train_processed.shape[1])
log_train_results = metrics_table(model_names, y_train, train_preds, X_train_processed.shape[1])
log_test_results = metrics_table(model_names, y_test, test_preds, X_train_processed.shape[1])

print('Train metrics (original price scale):')
display(train_results)
print('Test metrics (original price scale):')
display(test_results)
print('Train metrics (log-price scale):')
display(log_train_results)
print('Test metrics (log-price scale):')
display(log_test_results)


Train metrics (original price scale):


,model,mae,mse,rmse,mape,r2,adj_r2
0,Linear Regression,70.123955,312056.727683,558.620379,42.880635,-8.572801,-8.584145
1,Ridge Regression,70.122528,311908.012850,558.487254,42.879203,-8.568239,-8.579577
2,Lasso Regression,70.264922,325452.271198,570.484243,43.020059,-8.983729,-8.995560
3,Polynomial Ridge,52.281275,17451.303353,132.103381,33.372125,0.464656,0.464021
4,HistGradientBoosting,43.923464,6828.476001,82.634593,28.329546,0.790526,0.790278
5,XGBoost,37.319410,4864.505469,69.746007,24.585905,0.850774,0.850597
6,LightGBM,39.454733,5497.610941,74.145876,25.718486,0.831353,0.831153
7,CatBoost,43.412313,6558.434625,80.984163,28.059427,0.798810,0.798572


Test metrics (original price scale):


,model,mae,mse,rmse,mape,r2,adj_r2
0,Linear Regression,7.080253e+01,1.355160e+05,3.681250e+02,4.048845e+01,-2.953251e+00,-2.972057e+00
1,Ridge Regression,7.080073e+01,1.354319e+05,3.680108e+02,4.048763e+01,-2.950798e+00,-2.969593e+00
2,Lasso Regression,7.076086e+01,1.284050e+05,3.583365e+02,4.059009e+01,-2.745811e+00,-2.763630e+00
3,Polynomial Ridge,1.674677e+14,2.310384e+32,1.519995e+16,7.443011e+13,-6.739814e+27,-6.771877e+27
4,HistGradientBoosting,4.539036e+01,7.361449e+03,8.579889e+01,2.955041e+01,7.852530e-01,7.842314e-01
5,XGBoost,4.311229e+01,6.735604e+03,8.207072e+01,2.843847e+01,8.035101e-01,8.025753e-01
6,LightGBM,4.310520e+01,6.600882e+03,8.124581e+01,2.845307e+01,8.074402e-01,8.065241e-01
7,CatBoost,4.470995e+01,6.994406e+03,8.363256e+01,2.935569e+01,7.959603e-01,7.949897e-01


Train metrics (log-price scale):


,model,mae,mse,rmse,mape,r2,adj_r2
0,Linear Regression,0.351298,0.215448,0.464164,7.647859,0.712722,0.712382
1,Ridge Regression,0.351304,0.215449,0.464165,7.647988,0.712721,0.712380
2,Lasso Regression,0.352573,0.216220,0.464994,7.677457,0.711693,0.711352
3,Polynomial Ridge,0.300842,0.157881,0.397343,6.573501,0.789482,0.789232
4,HistGradientBoosting,0.267471,0.124498,0.352842,5.869868,0.833995,0.833799
5,XGBoost,0.234380,0.097544,0.312320,5.165649,0.869935,0.869781
6,LightGBM,0.244413,0.105507,0.324818,5.379548,0.859318,0.859151
7,CatBoost,0.265027,0.121936,0.349193,5.817285,0.837411,0.837219


Test metrics (log-price scale):


,model,mae,mse,rmse,mape,r2,adj_r2
0,Linear Regression,0.345794,0.208733,0.456873,7.546691,0.726635,0.725335
1,Ridge Regression,0.345802,0.208736,0.456877,7.546840,0.726631,0.725330
2,Lasso Regression,0.347236,0.209690,0.457919,7.580468,0.725382,0.724076
3,Polynomial Ridge,0.304491,0.321091,0.566649,6.656588,0.579487,0.577487
4,HistGradientBoosting,0.275068,0.132413,0.363886,6.059249,0.826587,0.825762
5,XGBoost,0.264438,0.125730,0.354585,5.835907,0.835339,0.834555
6,LightGBM,0.264371,0.125017,0.353577,5.835171,0.836273,0.835494
7,CatBoost,0.272482,0.129890,0.360403,6.006239,0.829891,0.829081
